# 03 — Modelling

**Hospital Readmission Risk Predictor**

This notebook trains, tunes, calibrates, and evaluates models for 30-day readmission prediction.
We start with a Logistic Regression baseline, compare four advanced models via cross-validation,
calibrate the winner, and optimise the decision threshold for clinical use.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_and_prepare
from src.leakage_detection import remove_leakage_rows
from src.feature_engineering import engineer_features
from src.preprocessing import resolve_feature_lists, split_X_y
from src.train import train_baseline, train_all_models, select_best_model
from src.evaluate import evaluate_model, compute_metrics, print_classification_report
from src.calibrate import calibrate_model, compare_calibration_methods, plot_calibration_curve, expected_calibration_error
from src.threshold_optimizer import find_optimal_threshold, plot_threshold_analysis, document_threshold_rationale
from src.utils import set_plot_style, save_pipeline, set_seed
from src.config import RANDOM_SEED, TEST_SIZE

from sklearn.model_selection import train_test_split
from sklearn.metrics import brier_score_loss

set_plot_style()
set_seed()
print('Setup complete ✓')

## 1. Data Preparation

In [ ]:
df = load_and_prepare()
df = remove_leakage_rows(df)
df = engineer_features(df)

X, y = split_X_y(df)
numeric_cols, categorical_cols = resolve_feature_lists(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)

print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')
print(f'Positive rate — train: {y_train.mean():.3f}  test: {y_test.mean():.3f}')

## 2. Baseline: Logistic Regression

Every modelling exercise should start with a simple, interpretable baseline. We use Logistic
Regression with `class_weight="balanced"` to handle the imbalance. This gives us a floor to
beat with more complex models.

In [ ]:
baseline = train_baseline(X_train, y_train, numeric_cols, categorical_cols)

baseline_eval = evaluate_model(baseline, X_test, y_test)
print('\nBaseline metrics:')
for k, v in baseline_eval['metrics'].items():
    print(f'  {k:12s} {v:.4f}')

### Baseline Analysis

The Logistic Regression with balanced class weights provides decent recall but typically at the
expense of precision. The ROC-AUC gives us a baseline discrimination ability to beat. Let's see
if tree-based models can do better.

## 3. Advanced Models — Hyperparameter Tuning

We train four candidates using `RandomizedSearchCV` with 5-fold cross-validation, optimising
ROC-AUC:

1. **GradientBoostingClassifier** — Sequential boosting, strong on tabular data
2. **RandomForestClassifier** — Bagging approach, naturally robust
3. **XGBoost** — Regularised gradient boosting, often state-of-the-art
4. **LightGBM** — Histogram-based boosting, fast and memory-efficient

In [ ]:
all_results = train_all_models(X_train, y_train, numeric_cols, categorical_cols)

# Summary table
comparison = pd.DataFrame({
    name: {'CV ROC-AUC': info['cv_score'], 'Train Time (s)': info['train_time_s']}
    for name, info in all_results.items()
}).T.sort_values('CV ROC-AUC', ascending=False)

print('\nModel Comparison:')
display(comparison)

In [ ]:
best_name, best_pipeline = select_best_model(all_results)
print(f'\nSelected model: {best_name}')

## 4. Evaluate Best Model on Test Set

In [ ]:
eval_result = evaluate_model(best_pipeline, X_test, y_test)

print(f'\n{best_name} Test Metrics:')
for k, v in eval_result['metrics'].items():
    print(f'  {k:12s} {v:.4f}')

## 5. Probability Calibration

Tree-based models rank patients well (good ROC-AUC) but their raw probability outputs can be
poorly calibrated. For clinical use, we need probabilities that actually mean what they say:
if we predict P(readmit) = 0.30, roughly 30% of similar patients should actually be readmitted.

We compare **sigmoid** (Platt scaling) and **isotonic** calibration.

In [ ]:
# Uncalibrated probabilities
y_prob_uncal = best_pipeline.predict_proba(X_test)[:, 1]
brier_uncal = brier_score_loss(y_test, y_prob_uncal)
ece_uncal = expected_calibration_error(y_test.values, y_prob_uncal)
print(f'Uncalibrated — Brier: {brier_uncal:.4f}  ECE: {ece_uncal:.4f}')

# Compare methods
best_method, calibrated_model = compare_calibration_methods(
    best_pipeline, X_train, y_train, X_test, y_test)

# Calibrated probabilities
y_prob_cal = calibrated_model.predict_proba(X_test)[:, 1]
brier_cal = brier_score_loss(y_test, y_prob_cal)
ece_cal = expected_calibration_error(y_test.values, y_prob_cal)

print(f'\nCalibrated ({best_method}) — Brier: {brier_cal:.4f}  ECE: {ece_cal:.4f}')
print(f'Brier improvement: {(brier_uncal - brier_cal)/brier_uncal*100:.1f}%')

In [ ]:
cal_path = plot_calibration_curve(y_test.values, y_prob_uncal, y_prob_cal)
print(f'Calibration curve saved: {cal_path}')

### Clinical Interpretation

Calibration is not just a statistical nicety — it has real clinical consequences. When a care
manager sees a predicted readmission probability of 40%, they need to trust that number to
make appropriate resource allocation decisions. A well-calibrated model enables:

- **Risk-stratified interventions** — different follow-up protocols for 10% vs 40% risk
- **Honest communication** with patients and families
- **Meaningful aggregate statistics** for hospital administration

## 6. Threshold Optimisation

The default 0.5 threshold is inappropriate for this problem. Missing a readmission (false negative)
costs far more than an unnecessary follow-up call (false positive). We find the threshold that
maximises recall while maintaining clinically reasonable precision.

In [ ]:
threshold_result = find_optimal_threshold(y_test.values, y_prob_cal, min_precision=0.15)

print('\nOptimal Threshold Analysis:')
for k, v in threshold_result.items():
    print(f'  {k:12s} {v:.4f}')

In [ ]:
plot_threshold_analysis(y_test.values, y_prob_cal, threshold_result['threshold'])
document_threshold_rationale(threshold_result)

In [ ]:
# Final evaluation at optimised threshold
final_eval = evaluate_model(calibrated_model, X_test, y_test,
                            threshold=threshold_result['threshold'])

print(f'\nFinal metrics at threshold={threshold_result["threshold"]:.3f}:')
for k, v in final_eval['metrics'].items():
    print(f'  {k:12s} {v:.4f}')

In [ ]:
# Save the calibrated pipeline
save_pipeline(calibrated_model)
print('Final calibrated pipeline saved as pipeline.pkl')

## Summary

| Stage | Key Result |
|-------|------------|
| Baseline (LR) | Decent recall with balanced weights; sets the floor |
| Model comparison | Tree-based models outperform LR on ROC-AUC |
| Calibration | Reduces Brier score; probabilities more reliable |
| Threshold | Lowered from 0.5 to maximise clinical recall |

The final model is a calibrated tree-based classifier with an optimised threshold,
ready for explainability analysis and fairness auditing.